In [1]:
import ee
ee.Authenticate()

ee.Initialize(project='fire-risk-agent')


In [2]:
from google.colab import userdata

GOOGLE_MAPS_KEY = userdata.get('GOOGLE_MAPS_KEY')

print("Key loaded:", GOOGLE_MAPS_KEY is not None)


Key loaded: True


In [3]:
!pip -q install requests
import requests

def geocode_google(address: str):
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    r = requests.get(url, params={
        "address": address,
        "key": GOOGLE_MAPS_KEY
    }, timeout=30)

    r.raise_for_status()
    data = r.json()

    if data["status"] != "OK":
        raise ValueError(f"Geocoding failed: {data['status']} - {data.get('error_message')}")

    loc = data["results"][0]["geometry"]["location"]
    return loc["lat"], loc["lng"]

#lat, lon = geocode_google("48978 River Park Rd, Oakhurst, CA")
#print(lat, lon)


In [4]:
#address = "48978 River Park Rd, Oakhurst, CA"
address = "17825 Woodcrest Dr, Pioneer, Ca"
def get_coordinates(address: str) -> dict:
    lat, lon = geocode_google(address)
    return {
        "address": address,
        "latitude": lat,
        "longitude": lon
    }
coords = get_coordinates(address)
lat = coords['latitude']
lon = coords['longitude']
print(lat, lon)

38.4655752 -120.5584229


In [5]:
def compute_mean_ndvi(lat, lon,
                      buffer_m=100,
                      start="2024-06-01",
                      end="2024-09-01",
                      cloud_pct=20):

    point = ee.Geometry.Point([lon, lat])
    area = point.buffer(buffer_m)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(area)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cloud_pct))
    )

    image = collection.median()
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")

    stats = ndvi.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=area,
        scale=10,
        maxPixels=1e9
    )

    return stats.get("NDVI").getInfo()


In [6]:
def classify_fuel(ndvi):
    if ndvi is None:
        return "No Data"
    elif ndvi >= 0.6:
        return "High Vegetation (High Fuel Load)"
    elif ndvi >= 0.3:
        return "Moderate Vegetation"
    elif ndvi >= 0.1:
        return "Sparse Vegetation"
    else:
        return "Minimal Vegetation"


In [7]:
ndvi_value = compute_mean_ndvi(lat, lon)
fuel_class = classify_fuel(ndvi_value)

result = {
    "latitude": lat,
    "longitude": lon,
    "mean_ndvi": ndvi_value,
    "fuel_class": fuel_class
}

result


{'latitude': 38.4655752,
 'longitude': -120.5584229,
 'mean_ndvi': 0.6418541604089351,
 'fuel_class': 'High Vegetation (High Fuel Load)'}

In [9]:
import geemap

Map = geemap.Map()
point = ee.Geometry.Point([lon, lat])
area = point.buffer(100)

image = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(area)
    .filterDate("2024-06-01", "2024-09-01")
    .median()
)

ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI").clip(area)

Map.centerObject(area, 17)
Map.addLayer(ndvi, {"min": 0, "max": 1}, "NDVI")
Map.addLayer(area, {}, "50m Buffer")
Map


Map(center=[38.46557530254494, -120.5584228962707], controls=(WidgetControl(options=['position', 'transparent_…